In [1]:
print("hi");


hi


In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10

# Optional SSL fix (if needed)
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

# 1. Load CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# 2. Normalize pixel values (0–255 → 0–1)
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# 3. Build CNN model
model = models.Sequential()

# Block 1
model.add(layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(32,32,3)))
model.add(layers.MaxPooling2D((2,2)))

# Block 2
model.add(layers.Conv2D(64, (3,3), activation='relu', padding='same'))
model.add(layers.MaxPooling2D((2,2)))

# Block 3
model.add(layers.Conv2D(128, (3,3), activation='relu', padding='same'))
model.add(layers.MaxPooling2D((2,2)))

# Flatten
model.add(layers.Flatten())

# Fully connected layers
model.add(layers.Dense(128, activation='relu'))
model.add(layers.Dropout(0.5))  # helps prevent overfitting

# Output layer (10 classes)
model.add(layers.Dense(10, activation='softmax'))

# 4. Compile model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 5. Train model
history = model.fit(
    x_train, y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1
)

# 6. Evaluate model
test_loss, test_acc = model.evaluate(x_test, y_test)

print("Test Accuracy:", test_acc)

170498071/170498071 [==============================] - 18s 0us/step
Epoch 1/10
704/704 [==============================] - 46s 62ms/step - loss: 1.6570 - accuracy: 0.3916 - val_loss: 1.2757 - val_accuracy: 0.5490
Epoch 2/10
704/704 [==============================] - 44s 63ms/step - loss: 1.2772 - accuracy: 0.5427 - val_loss: 1.0997 - val_accuracy: 0.6294
Epoch 3/10
704/704 [==============================] - 48s 68ms/step - loss: 1.1029 - accuracy: 0.6120 - val_loss: 0.9412 - val_accuracy: 0.6694
Epoch 4/10
704/704 [==============================] - 43s 61ms/step - loss: 0.9996 - accuracy: 0.6494 - val_loss: 0.8494 - val_accuracy: 0.7140
Epoch 5/10
704/704 [==============================] - 46s 66ms/step - loss: 0.9083 - accuracy: 0.6853 - val_loss: 0.8482 - val_accuracy: 0.7024
Epoch 6/10
704/704 [==============================] - 47s 66ms/step - loss: 0.8416 - accuracy: 0.7058 - val_loss: 0.8539 - val_accuracy: 0.7066
Epoch 7/10
704/704 [==============================] - 43s 61ms/step 

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.applications import ResNet50
import numpy as np

# 1. Setup Data
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# 2. Define Model Architectures

def get_lenet():
    return models.Sequential([
        layers.Conv2D(6, (5, 5), activation='relu', input_shape=(32, 32, 3)),
        layers.MaxPooling2D(),
        layers.Conv2D(16, (5, 5), activation='relu'),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(120, activation='relu'),
        layers.Dense(84, activation='relu'),
        layers.Dense(10, activation='softmax')
    ], name="LeNet-5")

def get_alexnet():
    # Modified for 32x32 input
    return models.Sequential([
        layers.Conv2D(48, (3, 3), padding='same', activation='relu', input_shape=(32, 32, 3)),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Conv2D(192, (3, 3), padding='same', activation='relu'),
        layers.Conv2D(192, (3, 3), padding='same', activation='relu'),
        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Flatten(),
        layers.Dense(1024, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1024, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax')
    ], name="AlexNet_Modified")

def get_vgg16():
    # Standard VGG16 blocks
    model = models.Sequential(name="VGG-16")
    cfg = [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 'M'] # Reduced depth for CIFAR
    model.add(layers.Input(shape=(32, 32, 3)))
    for v in cfg:
        if v == 'M':
            model.add(layers.MaxPooling2D((2, 2)))
        else:
            model.add(layers.Conv2D(v, (3, 3), padding='same', activation='relu'))
    model.add(layers.Flatten())
    model.add(layers.Dense(512, activation='relu'))
    model.add(layers.Dense(10, activation='softmax'))
    return model

def get_resnet50():
    # Using Keras built-in ResNet50
    base_resnet = ResNet50(include_top=False, weights=None, input_shape=(32, 32, 3))
    model = models.Sequential([
        base_resnet,
        layers.GlobalAveragePooling2D(),
        layers.Dense(10, activation='softmax')
    ], name="ResNet-50")
    return model

# 3. Training Loop
model_factories = [get_lenet, get_alexnet, get_vgg16, get_resnet50]
results = {}

for factory in model_factories:
    model = factory()
    print(f"\n--- Training {model.name} ---")
    
    model.compile(optimizer='adam', 
                  loss='sparse_categorical_crossentropy', 
                  metrics=['accuracy'])
    
    # Training for 5 epochs only for demonstration (Increase to 50+ for full accuracy)
    model.fit(x_train, y_train, epochs=5, batch_size=128, validation_split=0.1, verbose=1)
    
    loss, acc = model.evaluate(x_test, y_test, verbose=0)
    results[model.name] = acc
    print(f"{model.name} Test Accuracy: {acc:.4f}")

# 4. Final Comparison
print("\n" + "="*30)
print("FINAL ACCURACY COMPARISON")
print("="*30)
for name, acc in results.items():
    print(f"{name:20}: {acc*100:.2f}%")


--- Training LeNet-5 ---
Epoch 1/5
352/352 [==============================] - 13s 34ms/step - loss: 1.7634 - accuracy: 0.3612 - val_loss: 1.5044 - val_accuracy: 0.4564
Epoch 2/5
352/352 [==============================] - 12s 34ms/step - loss: 1.4581 - accuracy: 0.4765 - val_loss: 1.3734 - val_accuracy: 0.5082
Epoch 3/5
352/352 [==============================] - 12s 35ms/step - loss: 1.3346 - accuracy: 0.5237 - val_loss: 1.2879 - val_accuracy: 0.5414
Epoch 4/5
352/352 [==============================] - 13s 36ms/step - loss: 1.2576 - accuracy: 0.5520 - val_loss: 1.2143 - val_accuracy: 0.5706
Epoch 5/5
352/352 [==============================] - 13s 36ms/step - loss: 1.2052 - accuracy: 0.5729 - val_loss: 1.1908 - val_accuracy: 0.5832
LeNet-5 Test Accuracy: 0.5705

--- Training AlexNet_Modified ---
Epoch 1/5
352/352 [==============================] - 187s 531ms/step - loss: 1.6886 - accuracy: 0.3660 - val_loss: 1.2368 - val_accuracy: 0.5416
Epoch 2/5
352/352 [==============================